### Create awards from DataCite records where resourceTypeGeneral is "Award"

Mirrors `CreateCrossrefAwards` for DataCite. Strict filter: only records with `attributes.types.resourceTypeGeneral == 'Award'`. ESRF (`10.15151/`) records are out of scope here — they are `resourceTypeGeneral == 'Dataset'` and are handled by the work↔funder/award linkage job (#125.2).

In [ ]:
import dlt
import pyspark.sql.functions as F
from pyspark.sql.window import Window


INVESTIGATOR_STRUCT = (
    "struct<"
    "given_name:string,"
    "family_name:string,"
    "orcid:string,"
    "role_start:date,"
    "affiliation:struct<"
    "name:string,"
    "country:string,"
    "ids:array<struct<id:string,type:string,asserted_by:string>>"
    ">"
    ">"
)


def parse_datacite_person(person_col):
    """
    Parse a DataCite creator/contributor entry (nameType='Personal') into the
    OpenAlex investigator schema. Mirrors the shape produced by parse_investigator
    in CreateCrossrefAwards but sourced from DataCite fields.
    """
    return F.struct(
        person_col["givenName"].alias("given_name"),
        person_col["familyName"].alias("family_name"),
        F.when(
            F.get(person_col["nameIdentifiers"]["nameIdentifierScheme"], 0) == "ORCID",
            F.regexp_extract(
                F.get(person_col["nameIdentifiers"], 0)["nameIdentifier"],
                r"(\d{4}-\d{4}-\d{4}-\d{3}[\dXx])",
                1,
            ),
        ).alias("orcid"),
        # DataCite has no role-start equivalent
        F.lit(None).cast("date").alias("role_start"),
        F.when(
            F.size(person_col["affiliation"]) > 0,
            F.struct(
                person_col["affiliation"][0]["name"].alias("name"),
                F.lit(None).cast("string").alias("country"),
                F.when(
                    F.lower(person_col["affiliation"][0]["affiliationIdentifierScheme"]) == "ror",
                    F.array(
                        F.struct(
                            person_col["affiliation"][0]["affiliationIdentifier"].alias("id"),
                            F.lit("ROR").alias("type"),
                            F.lit(None).cast("string").alias("asserted_by"),
                        )
                    ),
                )
                .otherwise(
                    F.array().cast("array<struct<id:string,type:string,asserted_by:string>>")
                )
                .alias("ids"),
            ),
        ).alias("affiliation"),
    )


@dlt.table(name="datacite_award_items")
def datacite_award_items():
    return (
        spark.read.table("openalex.datacite.datacite_items")
        .filter(F.col("attributes.types.resourceTypeGeneral") == "Award")
    )


@dlt.table(name="datacite_award_items_deduplicated")
def datacite_award_items_deduplicated():
    df = dlt.read("datacite_award_items")
    window = Window.partitionBy("id").orderBy(F.col("attributes.updated").desc())
    return (
        df
        .withColumn("row_num", F.row_number().over(window))
        .filter(F.col("row_num") == 1)
        .drop("row_num")
    )


@dlt.table(name="datacite_awards")
def datacite_awards():
    df = dlt.read("datacite_award_items_deduplicated")
    df_funders = spark.read.table("openalex.common.funder")

    fund_ref_0 = F.col("attributes.fundingReferences").getItem(0)

    # Investigator candidates, in priority order (PLAN.md):
    # 1. contributors with research roles (Personal)
    # 2. contributors with Researcher (Personal)
    # 3. creators (Personal)
    contributors_pi = F.filter(
        F.col("attributes.contributors"),
        lambda c: (c["nameType"] == "Personal")
        & c["contributorType"].isin(
            "ProjectLeader", "ProjectManager", "PrincipalInvestigator"
        ),
    )
    contributors_researcher = F.filter(
        F.col("attributes.contributors"),
        lambda c: (c["nameType"] == "Personal") & (c["contributorType"] == "Researcher"),
    )
    creators_personal = F.filter(
        F.col("attributes.creators"),
        lambda c: c["nameType"] == "Personal",
    )

    lead_candidate = F.coalesce(
        F.get(contributors_pi, 0),
        F.get(contributors_researcher, 0),
        F.get(creators_personal, 0),
    )

    # Investigators array: union of Personal creators + research-role Personal contributors.
    # No dedup in v1 — documented limitation; downstream consumers can dedup if needed.
    contributors_research_all = F.filter(
        F.col("attributes.contributors"),
        lambda c: (c["nameType"] == "Personal")
        & c["contributorType"].isin(
            "ProjectLeader", "ProjectManager", "PrincipalInvestigator", "Researcher"
        ),
    )
    all_investigators_raw = F.concat(creators_personal, contributors_research_all)

    # Description: first Abstract or Other.
    description_value = F.get(
        F.filter(
            F.col("attributes.descriptions"),
            lambda d: d["descriptionType"].isin("Abstract", "Other"),
        ),
        0,
    )["description"]

    # Date helpers. DataCite supports range syntax "2020-01-01/2024-12-31".
    def first_date_of_type(date_type):
        return F.to_date(
            F.split(
                F.get(
                    F.filter(
                        F.col("attributes.dates"),
                        lambda d: d["dateType"] == date_type,
                    ),
                    0,
                )["date"],
                "/",
            ).getItem(0)
        )

    valid_range_split = F.split(
        F.get(
            F.filter(
                F.col("attributes.dates"),
                lambda d: d["dateType"] == "Valid",
            ),
            0,
        )["date"],
        "/",
    )
    end_date_value = F.when(F.size(valid_range_split) >= 2, F.to_date(valid_range_split.getItem(1)))

    # Funder identifier extraction from fundingReferences[0]
    join_ror_id = F.when(
        fund_ref_0["funderIdentifierType"] == "ROR", fund_ref_0["funderIdentifier"]
    )
    join_doi = F.when(
        fund_ref_0["funderIdentifierType"].isin("DOI", "Crossref Funder ID"),
        F.regexp_replace(fund_ref_0["funderIdentifier"], r"^https?://(dx\.)?doi\.org/", ""),
    )

    # Stage everything we need for the join + projection
    df_staged = df.select(
        "*",
        fund_ref_0.alias("fund_ref"),
        F.coalesce(
            fund_ref_0["awardNumber"],
            F.regexp_extract(F.col("id"), r"10\.\d+/(.+)", 1),
        ).alias("funder_award_id"),
        F.lower(F.col("attributes.publisher.name")).alias("publisher_lower"),
        join_ror_id.alias("join_ror_id"),
        join_doi.alias("join_doi"),
        lead_candidate.alias("lead_candidate"),
        all_investigators_raw.alias("all_investigators_raw"),
        description_value.alias("description"),
        F.coalesce(
            first_date_of_type("Issued"),
            first_date_of_type("Created"),
            F.to_date(F.col("attributes.created")),
        ).alias("start_date"),
        end_date_value.alias("end_date"),
    )

    df_funders_ready = df_funders.select(
        F.col("funder_id").alias("f_funder_id"),
        F.col("display_name").alias("f_display_name"),
        F.col("ror_id").alias("f_ror_id"),
        F.col("doi").alias("f_doi"),
        F.lower(F.col("display_name")).alias("f_display_name_lower"),
    )

    # ID hash key: prefer resolved funder_id, fall back to publisher name
    id_hash = (
        F.abs(
            F.xxhash64(
                F.concat(
                    F.coalesce(F.col("f_funder_id").cast("string"), F.col("publisher_lower")),
                    F.lit(":"),
                    F.lower(F.col("funder_award_id")),
                )
            )
        )
        % 9000000000
    )

    return (
        df_staged.join(
            F.broadcast(df_funders_ready),
            (
                ((F.col("join_doi").isNotNull()) & (F.col("join_doi") == F.col("f_doi")))
                | ((F.col("join_ror_id").isNotNull()) & (F.col("join_ror_id") == F.col("f_ror_id")))
                | (
                    F.col("join_doi").isNull()
                    & F.col("join_ror_id").isNull()
                    & (F.col("publisher_lower") == F.col("f_display_name_lower"))
                )
            ),
            "left",
        ).select(
            id_hash.alias("id"),
            F.col("attributes.titles").getItem(0)["title"].alias("display_name"),
            F.col("description"),
            F.col("f_funder_id").alias("funder_id"),
            F.col("funder_award_id"),
            F.lit(None).cast("double").alias("amount"),
            F.lit(None).cast("string").alias("currency"),
            F.struct(
                F.when(
                    F.col("f_funder_id").isNotNull(),
                    F.concat(F.lit("https://openalex.org/F"), F.col("f_funder_id")),
                ).alias("id"),
                F.coalesce(
                    F.col("f_display_name"),
                    F.col("fund_ref.funderName"),
                    F.col("attributes.publisher.name"),
                ).alias("display_name"),
                F.col("f_ror_id").alias("ror_id"),
                F.col("f_doi").alias("doi"),
            ).alias("funder"),
            F.lit(None).cast("string").alias("funding_type"),
            F.lit(None).cast("string").alias("funder_scheme"),
            F.lit("datacite").alias("provenance"),
            F.col("start_date"),
            F.col("end_date"),
            F.year(F.col("start_date")).alias("start_year"),
            F.year(F.col("end_date")).alias("end_year"),
            F.when(
                F.col("lead_candidate").isNotNull(),
                parse_datacite_person(F.col("lead_candidate")),
            ).alias("lead_investigator"),
            F.lit(None).cast(INVESTIGATOR_STRUCT).alias("co_lead_investigator"),
            F.transform(F.col("all_investigators_raw"), parse_datacite_person).alias("investigators"),
            F.col("attributes.url").alias("landing_page_url"),
            F.concat(F.lit("https://doi.org/"), F.lower(F.col("id"))).alias("doi"),
            F.concat(
                F.lit("https://api.openalex.org/works?filter=awards.id:G"),
                id_hash.cast("string"),
            ).alias("works_api_url"),
            F.to_timestamp(F.col("attributes.created")).alias("created_date"),
            F.to_timestamp(F.col("attributes.updated")).alias("updated_date"),
        )
    )


## Insert into openalex_awards_raw

The SQL insert lives in `InsertDataCiteAwardsToRaw.ipynb` and runs as a separate task downstream of this DLT pipeline (mirrors the Crossref pattern: `CreateCrossrefAwards` → `InsertCrossrefAwardsToRaw`).